<a href="https://colab.research.google.com/github/asmaa-2003/LSTM-Anomaly--detection/blob/main/Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE

In [ ]:
!wget -q https://zenodo.org/records/7766691/files/MetroPT2.csv?download=1 -O MetroPT2.csv
DATA_PATH = "MetroPT2.csv"

# قراءة كامل dataset (لو RAM يسمح)
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")
df.head()

,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Flowmeter,Motor_current,COMP,...,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses,gpsLat,gpsLong,gpsSpeed,gpsQuality
0,2022-04-28 12:33:29.120,-0.014,8.060,1.136,-0.020,8.066,57.125,0.25,4.9100,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,41.140176,-8.609454,22.0,1.0
1,2022-04-28 12:33:30.111,0.156,8.058,-0.020,-0.018,8.066,57.200,0.25,4.7325,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,41.140229,-8.609471,22.0,1.0
2,2022-04-28 12:33:31.102,1.094,8.058,-0.026,-0.018,8.066,57.150,0.25,4.9100,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,41.140283,-8.609489,22.0,1.0
3,2022-04-28 12:33:32.093,2.482,8.058,-0.026,-0.018,8.064,57.125,0.25,4.8200,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,41.140338,-8.609505,22.0,1.0
4,2022-04-28 12:33:33.084,3.756,8.058,-0.024,-0.018,8.066,57.075,0.25,4.9100,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,41.140393,-8.609521,23.0,1.0


In [ ]:
# نأخذ البيانات من 2022-06-01 فصاعدًا (يحتوي على anomalies)
df = df[df["timestamp"] >= "2022-06-01"]

# إضافة عمود is_anomaly
df["is_anomaly"] = 0
failure_windows = [
    ("2022-06-01", "2022-06-02"),
    ("2022-07-10", "2022-07-11")
]
for start, end in failure_windows:
    mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
    df.loc[mask, "is_anomaly"] = 1

# تحقق من عدد كل class
print(df["is_anomaly"].value_counts())

is_anomaly
0    4296385
1     161686
Name: count, dtype: int64


In [ ]:
feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if "is_anomaly" in feature_cols:
    feature_cols.remove("is_anomaly")

X = df[feature_cols].values
y = df["is_anomaly"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (4458071, 20)
y shape: (4458071,)


In [ ]:
sm = SMOTE(sampling_strategy=1.0, random_state=42)
X_res, y_res = sm.fit_resample(X, y)

print("After SMOTE - class distribution:", np.bincount(y_res))

After SMOTE - class distribution: [4296385 4296385]


In [ ]:
sm = SMOTE(sampling_strategy=1.0, random_state=42)
X_res, y_res = sm.fit_resample(X, y)

print("After SMOTE - class distribution:", np.bincount(y_res))

After SMOTE - class distribution: [4296385 4296385]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res
)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',  # مهم لتعلم anomalies
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=200,
                       n_jobs=-1, random_state=42)

In [ ]:
y_pred = rf.predict(X_test)

precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n📊 Random Forest Evaluation Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")



📊 Random Forest Evaluation Metrics:
Precision: 0.7845
Recall:    0.9709
F1 Score:  0.8678
